# Text Summarization with PEGASUS on Samsum Dataset

Welcome to this interactive tutorial on training a deep learning model for **abstractive text summarization**.

## What You Will Learn

1. **Loading and exploring** the Samsum dialogue dataset
2. **Understanding PEGASUS** - a state-of-the-art summarization model
3. **Baseline evaluation** - how well does the pre-trained model perform?
4. **ROUGE scores** - understanding summarization metrics in depth
5. **Fine-tuning** with Hugging Face's `DataCollator` and `Trainer`
6. **Post-training evaluation** - measuring improvement
7. **Generating summaries** from the test set

---

## Background: Extractive vs Abstractive Summarization

- **Extractive**: Selects and concatenates important sentences from the source text
- **Abstractive**: Generates new sentences that capture the meaning (like a human would)

PEGASUS is an **abstractive** model - it can paraphrase, combine ideas, and generate novel text.

## Setup

Make sure you have activated the `dl-sandbox` conda environment before running this notebook:

```bash
conda activate dl-sandbox
```

All required packages (transformers, datasets, evaluate, rouge_score, etc.) are already installed in the environment.

### Hugging Face Authentication (Required for Samsum Dataset)

The Samsum dataset is a **gated dataset** - you need to:

1. **Create a Hugging Face account** at https://huggingface.co/join
2. **Accept the dataset terms** at https://huggingface.co/datasets/samsum (click "Agree and access repository")
3. **Create an access token** at https://huggingface.co/settings/tokens
4. **Login from terminal** (run once before starting the notebook):

```bash
huggingface-cli login
```

Enter your token when prompted. This saves credentials locally so you won't need to do it again.

In [1]:
import os
# Disable tokenizers parallelism to avoid fork warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Disable wandb by default (can be enabled if you have an account)
os.environ["WANDB_DISABLED"] = "true"

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Determine the best available device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be slow)")

print(f"\nPyTorch version: {torch.__version__}")

Using Apple Silicon MPS

PyTorch version: 2.9.1


### Explanation: Device Selection

Deep learning models benefit greatly from hardware acceleration:
- **CUDA**: NVIDIA GPUs - fastest option for most models
- **MPS**: Apple Silicon (M1/M2/M3) - good performance on Mac
- **CPU**: Always available but significantly slower

The code above automatically selects the best available option.

---

## 1. Load the Samsum Dataset

**Samsum** is a dataset of ~16,000 messenger-like conversations with human-written summaries.

### Why Samsum?
- Dialogues are **informal** (slang, typos, emojis)
- Multiple **speakers** with turn-taking
- Requires understanding **context and intent**
- More challenging than news summarization

In [2]:
# Load the dataset
# Note: Requires Hugging Face authentication - see Setup section above
dataset = load_dataset("knkarthick/samsum", trust_remote_code=True)

print("Dataset Structure:")
print(dataset)
print(f"\nTraining samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['validation']):,}")
print(f"Test samples: {len(dataset['test']):,}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'knkarthick/samsum' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

Training samples: 14,731
Validation samples: 818
Test samples: 819


In [3]:
# Let's examine a sample
sample = dataset['train'][0]

print("=" * 60)
print("DIALOGUE:")
print("=" * 60)
print(sample['dialogue'])
print("\n" + "=" * 60)
print("HUMAN SUMMARY:")
print("=" * 60)
print(sample['summary'])

DIALOGUE:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

HUMAN SUMMARY:
Amanda baked cookies and will bring Jerry some tomorrow.


In [4]:
# Let's look at a few more examples to understand the data
print("Sample dialogues and their summaries:\n")
for i in [10, 50, 100]:
    print(f"--- Example {i} ---")
    print(f"Dialogue: {dataset['train'][i]['dialogue'][:200]}...")
    print(f"Summary: {dataset['train'][i]['summary']}")
    print()

Sample dialogues and their summaries:

--- Example 10 ---
Dialogue: Lucas: Hey! How was your day?
Demi: Hey there! 
Demi: It was pretty fine, actually, thank you!
Demi: I just got promoted! :D
Lucas: Whoa! Great news!
Lucas: Congratulations!
Lucas: Such a success has ...
Summary: Demi got promoted. She will celebrate that with Lucas at Death & Co at 10 pm.

--- Example 50 ---
Dialogue: Pitt: Hey Teddy! Have you received my message?
Teddy: No. An email?
Pitt: No. On the FB messenger.
Teddy: Let me check.
Teddy: Yeah. Ta!...
Summary: Teddy has a message from Pitt on Messenger.

--- Example 100 ---
Dialogue: Gabby: How is you? Settling into the new house OK?
Sandra: Good. The kids and the rest of the menagerie are doing fine. The dogs absolutely love the new garden. Plenty of room to dig and run around.
G...
Summary: Sandra is setting into the new house; her family is happy with it. Then Sandra and Gabby discuss the nature of their men and laugh about their habit of spending time in the g

### Explanation: Dataset Structure

Each sample contains:
- `id`: Unique identifier
- `dialogue`: The conversation text (input)
- `summary`: Human-written summary (target/label)

**Key Observations:**
- Dialogues have speaker names followed by colons
- Summaries are concise (1-2 sentences typically)
- Summaries capture the **action items** or **key decisions**

---

## 2. Understanding the Model

We'll use **FLAN-T5-Small** for this tutorial. While PEGASUS is excellent for summarization, it requires significant memory (570M parameters). FLAN-T5-Small (80M parameters) is much more memory-friendly while still demonstrating the same concepts.

### Why FLAN-T5?
- **Instruction-tuned**: Already trained to follow instructions like "summarize this"
- **Efficient**: Small model (80M params) fits easily in memory
- **Same architecture concepts**: Encoder-decoder transformer like PEGASUS
- **Great for learning**: Fast training iterations

### Model Variants (T5 Family)
- `google/flan-t5-small`: 80M parameters (we'll use this)
- `google/flan-t5-base`: 250M parameters
- `google/flan-t5-large`: 780M parameters
- `google/flan-t5-xl`: 3B parameters

For production use with more memory, you could use PEGASUS or larger T5 variants.

In [5]:
# Define model checkpoint - using smaller model for memory efficiency
model_checkpoint = "google/flan-t5-small"  # 80M params, memory-friendly

# Load tokenizer and model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Move model to device
model = model.to(device)

print(f"\nModel loaded successfully!")
print(f"Model: {model_checkpoint}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model is on: {next(model.parameters()).device}")

Loading tokenizer...
Loading model...

Model loaded successfully!
Model: google/flan-t5-small
Model parameters: 76,961,152
Model is on: mps:0


### Explanation: Tokenizer and Model

**Tokenizer**: Converts text to numerical tokens the model understands
- T5 uses SentencePiece tokenization (subword units)
- Handles unknown words by breaking them into subwords

**Model**: The neural network that does the actual summarization
- Encoder-Decoder architecture (like original Transformer)
- Encoder processes the input dialogue
- Decoder generates the summary token by token

**T5 vs PEGASUS**:
- Both are encoder-decoder transformers
- T5 was trained with a text-to-text objective ("translate X to Y", "summarize: X")
- FLAN-T5 is instruction-tuned, making it better at following prompts

In [6]:
# Let's see how the tokenizer works
sample_text = "Hello, how are you doing today?"
tokens = tokenizer(sample_text)

print(f"Original text: {sample_text}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Decoded back: {tokenizer.decode(tokens['input_ids'], skip_special_tokens=True)}")
print(f"\nIndividual tokens:")
for token_id in tokens['input_ids']:
    print(f"  {token_id} -> '{tokenizer.decode([token_id])}'")

Original text: Hello, how are you doing today?
Token IDs: [8774, 6, 149, 33, 25, 692, 469, 58, 1]
Decoded back: Hello, how are you doing today?

Individual tokens:
  8774 -> 'Hello'
  6 -> ','
  149 -> 'how'
  33 -> 'are'
  25 -> 'you'
  692 -> 'doing'
  469 -> 'today'
  58 -> '?'
  1 -> '</s>'


---

## 3. Baseline Evaluation (Before Fine-tuning)

Before fine-tuning, let's see how well the pre-trained model (trained on news) performs on dialogue summarization.

This gives us a **baseline** to measure improvement after fine-tuning.

In [7]:
def generate_summary(text, model, tokenizer, device, max_input_length=512, max_output_length=128):
    """
    Generate a summary for a given text.
    
    Args:
        text: Input text to summarize
        model: The seq2seq model
        tokenizer: The tokenizer
        device: torch device (cuda/mps/cpu)
        max_input_length: Maximum tokens for input
        max_output_length: Maximum tokens for output
    
    Returns:
        Generated summary string
    """
    # Tokenize input
    inputs = tokenizer(
        text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_length,
            num_beams=4,           # Beam search for better quality
            length_penalty=2.0,     # Encourage longer summaries
            early_stopping=True
        )
    
    # Decode and return
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [8]:
# Test on a few examples
print("Testing pre-trained model on dialogue summarization:\n")

for i in [0, 5, 10]:
    sample = dataset['test'][i]
    
    print(f"{'='*60}")
    print(f"Example {i}")
    print(f"{'='*60}")
    print(f"\nDIALOGUE:\n{sample['dialogue']}")
    print(f"\nHUMAN SUMMARY:\n{sample['summary']}")
    
    generated = generate_summary(sample['dialogue'], model, tokenizer, device)
    print(f"\nMODEL SUMMARY (before fine-tuning):\n{generated}")
    print()

Testing pre-trained model on dialogue summarization:

Example 0

DIALOGUE:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

HUMAN SUMMARY:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

MODEL SUMMARY (before fine-tuning):
Larry called Hannah last time they were at the park together. Hannah doesn't know Larry well. Larry called her last time they were at the park together.

Example 5

DIALOGUE:
Benjamin: Hey guys, what are we doing with the keys today?
Hilary: I've got them. Whoever wants them can meet me at lunchtime or after
Elliot: I'm ok. We're meeting for the drinks in the evening anyway and

### Explanation: Generation Parameters

When generating summaries, several parameters control the output:

- **`num_beams=4`**: Beam search considers 4 candidates at each step, picking the best overall sequence (not just greedy token-by-token)
- **`length_penalty=2.0`**: Values > 1 encourage longer outputs; < 1 encourages shorter
- **`early_stopping=True`**: Stop when all beams produce an end token

**Observation**: The pre-trained model may struggle with dialogues because:
1. It was trained on **news articles**, not conversations
2. Dialogue format (speaker turns) is unfamiliar
3. Informal language differs from news style

---

## 4. Understanding ROUGE Scores

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard metric for summarization.

### ROUGE Variants

| Metric | What it measures | Example |
|--------|------------------|--------|
| **ROUGE-1** | Unigram (single word) overlap | "the cat sat" vs "the dog sat" -> 2/3 overlap |
| **ROUGE-2** | Bigram (2-word) overlap | "the cat" vs "the dog" -> 0/1 overlap |
| **ROUGE-L** | Longest Common Subsequence | Captures sentence-level structure |
| **ROUGE-Lsum** | ROUGE-L for multi-sentence summaries | Handles newlines properly |

### Precision vs Recall vs F1

For each ROUGE variant:
- **Precision**: What fraction of the generated summary overlaps with reference?
- **Recall**: What fraction of the reference is covered by generated summary?
- **F1**: Harmonic mean of precision and recall (most commonly reported)

### Example Calculation

```
Reference: "The cat sat on the mat"
Generated: "The cat is on the mat"

ROUGE-1:
- Matching words: the, cat, on, the, mat (5 matches)
- Precision: 5/6 = 0.83 (6 words in generated)
- Recall: 5/6 = 0.83 (6 words in reference)
- F1: 0.83
```

In [9]:
# Load the ROUGE metric
rouge_metric = evaluate.load("rouge")

# Example to understand ROUGE
reference = ["The cat sat on the mat"]
generated = ["The cat is on the mat"]

scores = rouge_metric.compute(predictions=generated, references=reference)

print("ROUGE Score Example:")
print(f"Reference: {reference[0]}")
print(f"Generated: {generated[0]}")
print(f"\nScores:")
for metric, score in scores.items():
    print(f"  {metric}: {score:.4f}")

ROUGE Score Example:
Reference: The cat sat on the mat
Generated: The cat is on the mat

Scores:
  rouge1: 0.8333
  rouge2: 0.6000
  rougeL: 0.8333
  rougeLsum: 0.8333


In [10]:
# Another example showing the difference between metrics
reference2 = ["The quick brown fox jumps over the lazy dog"]
generated2 = ["A fast brown fox leaps over a sleepy dog"]

scores2 = rouge_metric.compute(predictions=generated2, references=reference2)

print("ROUGE Score - Paraphrased Example:")
print(f"Reference: {reference2[0]}")
print(f"Generated: {generated2[0]}")
print(f"\nScores:")
for metric, score in scores2.items():
    print(f"  {metric}: {score:.4f}")

print("\n** Notice: Semantically similar but low ROUGE due to different words! **")
print("This is a limitation of ROUGE - it measures word overlap, not meaning.")

ROUGE Score - Paraphrased Example:
Reference: The quick brown fox jumps over the lazy dog
Generated: A fast brown fox leaps over a sleepy dog

Scores:
  rouge1: 0.4444
  rouge2: 0.1250
  rougeL: 0.4444
  rougeLsum: 0.4444

** Notice: Semantically similar but low ROUGE due to different words! **
This is a limitation of ROUGE - it measures word overlap, not meaning.


### Explanation: Interpreting ROUGE Scores

**What's a good ROUGE score?**

It depends on the task and dataset:
- **News summarization (CNN/DM)**: ROUGE-1 ~40-45, ROUGE-2 ~17-21, ROUGE-L ~37-40
- **Dialogue summarization (Samsum)**: ROUGE-1 ~45-53, ROUGE-2 ~21-28, ROUGE-L ~40-45

**Key insights:**
1. ROUGE-2 is harder (requires exact 2-word matches)
2. Scores vary significantly by domain
3. ROUGE doesn't capture semantic similarity well
4. Always compare models on the SAME dataset

---

## 5. Evaluate Pre-trained Model on Test Set

Now let's compute ROUGE scores on the full test set to establish our baseline.

In [11]:
def evaluate_summaries_pegasus(
    dataset_split,
    model,
    tokenizer,
    device,
    num_samples=100,
    batch_size=4
):
    """
    Evaluate PEGASUS model on a dataset split using ROUGE scores.
    
    Args:
        dataset_split: HuggingFace dataset split (e.g., dataset['test'])
        model: The seq2seq model
        tokenizer: The tokenizer
        device: torch device
        num_samples: Number of samples to evaluate (for speed)
        batch_size: Batch size for generation
    
    Returns:
        Dictionary of ROUGE scores
    """
    rouge_metric = evaluate.load("rouge")
    
    # Sample if needed
    if num_samples and num_samples < len(dataset_split):
        indices = list(range(num_samples))
        samples = dataset_split.select(indices)
    else:
        samples = dataset_split
    
    generated_summaries = []
    reference_summaries = []
    
    model.eval()
    
    # Process in batches for efficiency
    for i in tqdm(range(0, len(samples), batch_size), desc="Generating summaries"):
        batch = samples[i:i + batch_size]
        dialogues = batch['dialogue']
        references = batch['summary']
        
        # Tokenize batch
        inputs = tokenizer(
            dialogues,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)
        
        # Generate
        with torch.no_grad():
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=128,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True
            )
        
        # Decode
        decoded = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        
        generated_summaries.extend(decoded)
        reference_summaries.extend(references)
    
    # Compute ROUGE
    results = rouge_metric.compute(
        predictions=generated_summaries,
        references=reference_summaries
    )
    
    return results, generated_summaries, reference_summaries

In [12]:
# Evaluate baseline model (using 100 samples for speed)
print("Evaluating pre-trained model on test set...")
print("(Using 100 samples for faster evaluation)\n")

baseline_results, baseline_generated, baseline_references = evaluate_summaries_pegasus(
    dataset['test'],
    model,
    tokenizer,
    device,
    num_samples=100
)

print("\n" + "="*50)
print("BASELINE ROUGE SCORES (Before Fine-tuning)")
print("="*50)
for metric, score in baseline_results.items():
    print(f"{metric}: {score:.4f}")

Evaluating pre-trained model on test set...
(Using 100 samples for faster evaluation)



Generating summaries:   0%|          | 0/25 [00:00<?, ?it/s]


BASELINE ROUGE SCORES (Before Fine-tuning)
rouge1: 0.4260
rouge2: 0.1778
rougeL: 0.3472
rougeLsum: 0.3482


### Explanation: Baseline Performance

The pre-trained PEGASUS model (trained on CNN/DailyMail news) typically achieves:
- Lower ROUGE scores on dialogue data
- This is expected due to **domain mismatch**

Fine-tuning on Samsum will teach the model:
1. Dialogue structure (speaker turns)
2. Informal language patterns
3. What makes a good dialogue summary

---

## 6. Data Preprocessing for Fine-tuning

Before training, we need to:
1. Tokenize all dialogues and summaries
2. Set up labels (target summaries)
3. Handle padding and truncation

In [13]:
# Define max lengths
MAX_INPUT_LENGTH = 512   # Max tokens for dialogue
MAX_TARGET_LENGTH = 128  # Max tokens for summary

def preprocess_function(examples):
    """
    Tokenize dialogues and summaries for training.
    
    This function:
    1. Tokenizes the input dialogues
    2. Tokenizes the target summaries (as labels)
    3. Handles truncation for long texts
    """
    # Tokenize inputs (dialogues)
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        # Note: We don't pad here - DataCollator will handle dynamic padding
    )
    
    # Tokenize targets (summaries)
    # We use text_target parameter to indicate these are target sequences
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [14]:
# Apply preprocessing to the entire dataset
print("Tokenizing dataset...")

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,  # Process in batches for speed
    remove_columns=dataset["train"].column_names,  # Remove original columns
    desc="Tokenizing"
)

print(f"\nTokenized dataset structure:")
print(tokenized_dataset)
print(f"\nSample tokenized entry keys: {tokenized_dataset['train'][0].keys()}")

Tokenizing dataset...


Tokenizing:   0%|          | 0/818 [00:00<?, ? examples/s]


Tokenized dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

Sample tokenized entry keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [15]:
# Let's examine what the tokenized data looks like
sample_tokenized = tokenized_dataset['train'][0]

print("Tokenized Sample:")
print(f"Input IDs length: {len(sample_tokenized['input_ids'])}")
print(f"Labels length: {len(sample_tokenized['labels'])}")
print(f"\nDecoded input (first 200 chars):")
print(tokenizer.decode(sample_tokenized['input_ids'])[:200])
print(f"\nDecoded labels:")
print(tokenizer.decode(sample_tokenized['labels']))

Tokenized Sample:
Input IDs length: 28
Labels length: 11

Decoded input (first 200 chars):
Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)</s>

Decoded labels:
Amanda baked cookies and will bring Jerry some tomorrow.</s>


### Explanation: Why No Padding Yet?

We don't pad during preprocessing because:
1. **Dynamic padding** is more efficient
2. The `DataCollatorForSeq2Seq` pads each batch to the max length in that batch
3. This avoids wasting computation on padding tokens

For example:
- Batch 1 might have max length 256 -> pad to 256
- Batch 2 might have max length 128 -> pad to 128

This is more efficient than padding everything to 512!

---

## 7. Setting Up the DataCollator

The **DataCollator** handles:
- Dynamic padding to batch max length
- Creating attention masks
- Setting padding tokens in labels to -100 (ignored by loss function)

In [16]:
# Create the data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # Enable dynamic padding
    label_pad_token_id=-100  # Ignore padding in loss calculation
)

print("DataCollator created!")
print(f"Padding token ID: {tokenizer.pad_token_id}")
print(f"Label padding token ID: -100 (ignored in loss)")

DataCollator created!
Padding token ID: 0
Label padding token ID: -100 (ignored in loss)


In [17]:
# Let's see what the data collator produces
sample_batch = [tokenized_dataset['train'][i] for i in range(3)]
collated = data_collator(sample_batch)

print("Collated batch:")
for key, value in collated.items():
    print(f"  {key}: shape {value.shape}")

print(f"\nNote: All samples padded to same length within the batch!")

Collated batch:
  input_ids: shape torch.Size([3, 152])
  attention_mask: shape torch.Size([3, 152])
  labels: shape torch.Size([3, 19])
  decoder_input_ids: shape torch.Size([3, 19])

Note: All samples padded to same length within the batch!


### Explanation: Label Padding with -100

Why do we use `-100` for label padding?

```
Labels:    [   5,  42, 103, -100, -100]  <- -100 marks padding
                              |
CrossEntropyLoss ignores positions with -100
                              |
Model only learns from real tokens, not padding!
```

This is a PyTorch convention - `CrossEntropyLoss(ignore_index=-100)` is the default.

---

## 8. Define ROUGE Metric for Training

We need a function that computes ROUGE during evaluation steps of training.

In [18]:
# Load ROUGE metric
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """
    Compute ROUGE metrics during training evaluation.
    
    Args:
        eval_pred: Tuple of (predictions, labels) from Trainer
    
    Returns:
        Dictionary of ROUGE scores
    """
    predictions, labels = eval_pred
    
    # Replace -100 and any negative values in predictions with pad_token_id
    # This handles the padding that can cause overflow errors
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in labels (padding) with pad_token_id for decoding
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute ROUGE
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    return result

---

## 9. Configure Training with Seq2SeqTrainer

The Hugging Face `Trainer` API simplifies training by handling:
- Training loop
- Gradient accumulation
- Learning rate scheduling
- Checkpointing
- Evaluation
- Logging

In [19]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/flan-t5-samsum",  # Where to save checkpoints
    
    # Training hyperparameters
    num_train_epochs=3,              # Number of epochs
    per_device_train_batch_size=4,   # Batch size (can be larger with smaller model)
    per_device_eval_batch_size=4,    # Eval batch size
    gradient_accumulation_steps=2,   # Effective batch = 4*2=8
    
    # Learning rate
    learning_rate=3e-4,              # Slightly higher LR works well for T5
    warmup_steps=100,                # Warmup for stability
    weight_decay=0.01,               # Regularization
    
    # Evaluation and saving
    eval_strategy="steps",
    eval_steps=500,                  # Evaluate every 500 steps
    save_strategy="steps",
    save_steps=500,                  # Save checkpoint every 500 steps
    save_total_limit=2,              # Keep only 2 best checkpoints
    load_best_model_at_end=True,     # Load best model when done
    metric_for_best_model="rouge1",  # Use ROUGE-1 to determine best
    greater_is_better=True,
    
    # Generation settings for evaluation
    predict_with_generate=True,      # Enable generation during eval
    generation_max_length=128,
    
    # Logging
    logging_dir="../models/flan-t5-samsum/logs",
    logging_steps=100,
    report_to="none",                # Disable wandb/tensorboard reporting
    
    # Performance settings for MPS
    fp16=False,                      # MPS doesn't support fp16 well
    dataloader_num_workers=0,        # Required for MPS
    
    # Reproducibility
    seed=42,
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")

Training configuration:
  Epochs: 3
  Batch size: 4
  Gradient accumulation: 2
  Effective batch size: 8
  Learning rate: 0.0003


### Explanation: Training Hyperparameters

**Batch Size & Gradient Accumulation:**
- PEGASUS is large, so we use small batch size (2) to fit in memory
- `gradient_accumulation_steps=4` simulates larger batches
- Effective batch size = 2 x 4 = 8

**Learning Rate:**
- `5e-5` is standard for fine-tuning pre-trained models
- Too high -> unstable training, forget pre-training
- Too low -> slow learning

**Warmup:**
- Gradually increase LR at start
- Prevents large updates that could destabilize early training

**Weight Decay:**
- L2 regularization to prevent overfitting
- 0.01 is a common value

In [20]:
# Reload fresh model for training
print("Reloading fresh model for training...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

# Clear any cached memory
if device.type == "mps":
    torch.mps.empty_cache()

# Create the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully!")
print(f"Training samples: {len(tokenized_dataset['train']):,}")
print(f"Validation samples: {len(tokenized_dataset['validation']):,}")

Reloading fresh model for training...
Trainer created successfully!
Training samples: 14,731
Validation samples: 818


## 10. Fine-tune the Model

Now we train! This will take a while depending on your hardware:
- **GPU**: ~30-60 minutes
- **MPS (Apple Silicon)**: ~1-2 hours  
- **CPU**: Several hours (not recommended)

**Tip:** You can reduce `num_train_epochs` to 1 for a quick test.

In [21]:
# Start training!
print("Starting fine-tuning...")
print("This may take a while. Training progress will be shown below.\n")

train_result = trainer.train()

print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"Total training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

Starting fine-tuning...
This may take a while. Training progress will be shown below.



Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
500,1.874600,1.716270,0.439674,0.201931,0.366786,0.366165
1000,1.818500,1.685714,0.433556,0.203360,0.362688,0.362218
1500,1.804700,1.665897,0.447359,0.212490,0.375973,0.375688
2000,1.638800,1.648294,0.452764,0.212744,0.373075,0.372902
2500,1.657500,1.639283,0.457399,0.218608,0.378308,0.378148
3000,1.599400,1.631480,0.456511,0.216234,0.380523,0.380170
3500,1.622500,1.612098,0.452025,0.215840,0.378653,0.378505
4000,1.476400,1.625397,0.458035,0.221247,0.385453,0.385478
4500,1.485200,1.621603,0.461954,0.224348,0.383574,0.383487
5000,1.550200,1.611871,0.463911,0.225848,0.386806,0.386198


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



Training Complete!
Total training time: 2529.36 seconds
Training samples/second: 17.47


In [22]:
# Save the final model
trainer.save_model("../models/flan-t5-samsum-final")
tokenizer.save_pretrained("../models/flan-t5-samsum-final")

print("Model saved to ../models/flan-t5-samsum-final")

Model saved to ../models/flan-t5-samsum-final


### Explanation: What Happened During Training?

Each training step:
1. **Forward pass**: Model generates summaries, compares to targets
2. **Loss calculation**: Cross-entropy loss measures prediction error
3. **Backward pass**: Gradients computed via backpropagation
4. **Optimizer step**: Weights updated to reduce loss

The model learns:
- Dialogue-specific patterns
- How to summarize conversations
- When to include speaker names vs. just actions

---

## 11. Evaluate Fine-tuned Model

Let's see how much we improved over the baseline!

In [23]:
# Evaluate on the same test samples as baseline
print("Evaluating fine-tuned model on test set...\n")

finetuned_results, finetuned_generated, finetuned_references = evaluate_summaries_pegasus(
    dataset['test'],
    model,
    tokenizer,
    device,
    num_samples=100
)

print("\n" + "="*60)
print("COMPARISON: Before vs After Fine-tuning")
print("="*60)
print(f"{'Metric':<15} {'Baseline':>12} {'Fine-tuned':>12} {'Improvement':>12}")
print("-"*60)
for metric in baseline_results.keys():
    baseline_score = baseline_results[metric]
    finetuned_score = finetuned_results[metric]
    improvement = finetuned_score - baseline_score
    print(f"{metric:<15} {baseline_score:>12.4f} {finetuned_score:>12.4f} {improvement:>+12.4f}")

Evaluating fine-tuned model on test set...



Generating summaries:   0%|          | 0/25 [00:00<?, ?it/s]


COMPARISON: Before vs After Fine-tuning
Metric              Baseline   Fine-tuned  Improvement
------------------------------------------------------------
rouge1                0.4260       0.4626      +0.0366
rouge2                0.1778       0.2093      +0.0315
rougeL                0.3472       0.3832      +0.0359
rougeLsum             0.3482       0.3845      +0.0364


### Explanation: Interpreting the Improvement

After fine-tuning, you should see improvements across all ROUGE metrics:

- **ROUGE-1 improvement**: Better word overlap (vocabulary adaptation)
- **ROUGE-2 improvement**: Better phrase matching (learned dialogue patterns)
- **ROUGE-L improvement**: Better overall structure (learned summary style)

The model has adapted from news summarization to dialogue summarization!

---

## 12. Generating Dialogue Summaries from Test Set

Let's see the fine-tuned model in action on test examples!

In [24]:
# Compare summaries side by side
print("Side-by-side comparison of summaries:\n")

for i in [0, 5, 10, 15, 20]:
    sample = dataset['test'][i]
    
    # Generate with fine-tuned model
    generated = generate_summary(sample['dialogue'], model, tokenizer, device)
    
    print(f"{'='*70}")
    print(f"TEST EXAMPLE {i}")
    print(f"{'='*70}")
    print(f"\nDIALOGUE:")
    print(sample['dialogue'])
    print(f"\nREFERENCE SUMMARY:")
    print(sample['summary'])
    print(f"\nMODEL GENERATED SUMMARY:")
    print(generated)
    
    # Compute ROUGE for this example
    example_rouge = rouge_metric.compute(
        predictions=[generated],
        references=[sample['summary']]
    )
    print(f"\nROUGE Scores: R1={example_rouge['rouge1']:.3f}, R2={example_rouge['rouge2']:.3f}, RL={example_rouge['rougeL']:.3f}")
    print()

Side-by-side comparison of summaries:

TEST EXAMPLE 0

DIALOGUE:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

REFERENCE SUMMARY:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

MODEL GENERATED SUMMARY:
Amanda can't find Betty's number. Larry called her last time they were at the park together.

ROUGE Scores: R1=0.353, R2=0.125, RL=0.235

TEST EXAMPLE 5

DIALOGUE:
Benjamin: Hey guys, what are we doing with the keys today?
Hilary: I've got them. Whoever wants them can meet me at lunchtime or after
Elliot: I'm ok. We're meeting for the drinks in the evening anyway and I guess we'll be going back

In [25]:
# Create a summary comparison table
print("Generating summaries for comparison table...\n")

comparison_data = []
for i in range(10):
    sample = dataset['test'][i]
    generated = generate_summary(sample['dialogue'], model, tokenizer, device)
    comparison_data.append({
        'Dialogue (truncated)': sample['dialogue'][:100] + "...",
        'Reference': sample['summary'],
        'Generated': generated
    })

results_df = pd.DataFrame(comparison_data)
print("Summary comparison table:")
results_df

Generating summaries for comparison table...

Summary comparison table:


,Dialogue (truncated),Reference,Generated
0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,Amanda can't find Betty's number. Larry called...
1,Eric: MACHINE!\nRob: That's so gr8!\nEric: I k...,Eric and Rob are going to watch a stand-up on ...,Rob and Eric are watching a stand-up on YouTube.
2,"Lenny: Babe, can you help me with something?\n...",Lenny can't decide which trousers to buy. Bob ...,Bob will help Lenny with choosing a pair of pu...
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,Emma will be home soon. Will will pick her up.
4,"Ollie: Hi , are you in Warsaw\nJane: yes, just...",Jane is in Warsaw. Ollie and Jane has a party....,Ollie is in Warsaw and Jane is free for dinner...
5,"Benjamin: Hey guys, what are we doing with the...",Hilary has the keys to the apartment. Benjamin...,"Benjamin, Hilary, Elliot and Daniel will meet ..."
6,Max: Know any good sites to buy clothes from?\...,Payton provides Max with websites selling clot...,Payton is looking for a place to buy clothes. ...
7,Rita: I'm so bloody tired. Falling asleep at w...,Rita and Tina are bored at work and have still...,Rita is tired at work. She keeps on looking at...
8,"Beatrice: I am in town, shopping. They have ni...","Beatrice wants to buy Leo a scarf, but he does...",Beatrice is in town and wants to buy a scarf. ...
9,Ivan: hey eric\nEric: yeah man\nIvan: so youre...,Eric doesn't know if his parents let him go to...,Eric is coming to the wedding with his brother...


### Explanation: Analyzing Generated Summaries

When reviewing the generated summaries, look for:

**Good signs:**
- Captures the main point of the conversation
- Grammatically correct
- Appropriate length
- Mentions key actors/actions

**Potential issues:**
- **Hallucination**: Generating facts not in the dialogue
- **Missing information**: Key points omitted
- **Repetition**: Same phrase repeated
- **Generic output**: Too vague to be useful

---

## 13. Try Your Own Dialogue!

Test the model on a custom conversation:

In [ ]:
# Custom dialogue example
custom_dialogue = """
Alice: Hey, are you coming to the meeting tomorrow?
Bob: What time is it?
Alice: 10 AM in the main conference room.
Bob: I have a doctor's appointment at 9:30. I might be late.
Alice: No worries, we can start with the budget review first. You can join for the product discussion.
Bob: Perfect, I should be there by 10:30.
Alice: Great, see you then!
"""

summary = generate_summary(custom_dialogue, model, tokenizer, device)

print("CUSTOM DIALOGUE:")
print(custom_dialogue)
print("\nGENERATED SUMMARY:")
print(summary)

In [ ]:
# Try another example - informal chat
casual_dialogue = """
Mike: dude did u see the game last night??
Jake: no man i fell asleep lol
Mike: u missed out! It was insane. Overtime and everything
Jake: who won?
Mike: Lakers by 2 points. LeBron hit a buzzer beater
Jake: no way!! gotta watch the highlights
Mike: ill send u the link
"""

summary2 = generate_summary(casual_dialogue, model, tokenizer, device)

print("CASUAL DIALOGUE:")
print(casual_dialogue)
print("\nGENERATED SUMMARY:")
print(summary2)

---

## 14. Loading the Saved Model

Here's how to load your fine-tuned model later:

In [ ]:
# Example: Loading saved model (run this in a new session)
print("To load the saved model in a new session, use this code:")
print()
print("""
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_path = "../models/flan-t5-samsum-final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Move to device
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)

# Generate a summary
dialogue = "Your dialogue here..."
inputs = tokenizer(dialogue, return_tensors="pt", max_length=512, truncation=True).to(device)
summary_ids = model.generate(inputs["input_ids"], max_length=128, num_beams=4)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)
""")

---

## Summary: What We Learned

### Key Concepts

1. **Abstractive Summarization**: Generating new text (not just extracting sentences)

2. **Seq2Seq Architecture**: 
   - Encoder-decoder transformer
   - Encoder processes the input dialogue
   - Decoder generates the summary token by token

3. **ROUGE Metrics**:
   - **ROUGE-1**: Unigram (word) overlap - measures vocabulary match
   - **ROUGE-2**: Bigram overlap - measures phrase-level match  
   - **ROUGE-L**: Longest common subsequence - measures fluency/structure
   - Higher is better; always compare on the same dataset
   - Limitation: doesn't capture semantic similarity

4. **Fine-tuning Process**:
   - Tokenize data with proper input/label handling
   - Use `DataCollatorForSeq2Seq` for dynamic padding
   - Configure `Seq2SeqTrainer` with appropriate hyperparameters
   - Monitor ROUGE during training for model selection

5. **Domain Adaptation**: 
   - Pre-trained models can adapt to new domains (dialogue)
   - Fine-tuning is more efficient than training from scratch

6. **Memory Considerations**:
   - Large models (PEGASUS 570M) may not fit on consumer hardware
   - Smaller models (FLAN-T5-Small 80M) work well for learning
   - Techniques: gradient checkpointing, reduced batch size, gradient accumulation

### Files Created
- `../models/flan-t5-samsum/` - Training checkpoints
- `../models/flan-t5-samsum-final/` - Final fine-tuned model

### Next Steps

- Try larger models if you have more memory: `google/flan-t5-base`, `google/pegasus-cnn_dailymail`
- Experiment with hyperparameters: learning rate, epochs, batch size
- Explore other datasets: XSum, CNN/DM, Multi-News
- Add semantic evaluation: BERTScore, BLEURT
- Deploy the model as an API using FastAPI or Gradio

In [ ]:
print("Congratulations! You've completed the PEGASUS summarization tutorial!")
print("\nYou learned how to:")
print("  - Load and explore the Samsum dialogue dataset")
print("  - Understand and use the PEGASUS model")
print("  - Evaluate models with ROUGE scores")
print("  - Fine-tune using DataCollator and Trainer")
print("  - Generate and analyze dialogue summaries")